In [ ]:
import time
import random
import logging
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import WebDriverException

In [18]:

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# YouTube channel URL
channel_url = "https://www.youtube.com/channel/UC1E-JS8L0j1Ei70D9VEFrPQ"

# Function to get all Wayback Machine snapshots
def get_all_snapshots(channel_url):
    api_url = f"http://web.archive.org/cdx/search/cdx?url={channel_url}&output=json&fl=timestamp,original&collapse=digest"
    response = requests.get(api_url)
    snapshots = response.json()
    return snapshots[1:]  # Skip the header

# Function to set up Selenium WebDriver
def setup_driver():
    options = Options()
    options.headless = True  # Run in headless mode (no browser window)
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

# Function to scrape subscriber count from snapshot using Selenium
def scrape_subscriber_count(driver, snapshot_url):
    retries = 5
    for i in range(retries):
        try:
            driver.get(snapshot_url)
            time.sleep(5)  # Wait for the page to load

            subscriber_count = None

            # Newer version of the page
            try:
                sub_count_element_new = driver.find_element(By.ID, "subscriber-count")
                if sub_count_element_new:
                    subscriber_count = sub_count_element_new.text.strip()
            except:
                pass

            # Older version of the page
            if not subscriber_count:
                try:
                    sub_count_element_old = driver.find_element(By.CSS_SELECTOR, "span.yt-subscription-button-subscriber-count-branded-horizontal.subscribed.yt-uix-tooltip")
                    if sub_count_element_old:
                        subscriber_count = sub_count_element_old.get_attribute("title").strip()
                except:
                    pass

            return subscriber_count
        except WebDriverException as e:
            logging.error(f"Error scraping {snapshot_url}: {e}")
            if 'disconnected' in str(e):
                return None
            time.sleep(2 ** i + random.uniform(0, 1))  # Exponential backoff
    return None

# Initialize WebDriver
driver = setup_driver()

# Collecting data
data = []

snapshots = get_all_snapshots(channel_url)
logging.info(f"Found {len(snapshots)} snapshots for the channel URL")

for snapshot in snapshots:
    timestamp = snapshot[0]
    original_url = snapshot[1]
    snapshot_url = f"https://web.archive.org/web/{timestamp}/{original_url}"
    subscriber_count = scrape_subscriber_count(driver, snapshot_url)
    
    # If we encounter a disconnection error, restart the driver
    if subscriber_count is None:
        driver.quit()
        driver = setup_driver()
        subscriber_count = scrape_subscriber_count(driver, snapshot_url)
    
    data.append({
        "snapshot_url": snapshot_url,
        "timestamp": timestamp,
        "subscriber_count": subscriber_count
    })
    logging.info(f"Scraped subscriber count for snapshot: {snapshot_url}")
    time.sleep(random.uniform(1, 5))  # Delay between requests to respect rate limits

# Quit the WebDriver
driver.quit()

# Convert data to DataFrame for easier analysis
df = pd.DataFrame(data)
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y%m%d%H%M%S')
print(df)

# Save to CSV
df.to_csv("channel_subscriber_counts_all_snapshots.csv", index=False)

2024-08-03 10:25:20,590 - INFO - ====== WebDriver manager ======
2024-08-03 10:25:20,838 - INFO - Get LATEST chromedriver version for google-chrome
2024-08-03 10:25:20,889 - INFO - Get LATEST chromedriver version for google-chrome
2024-08-03 10:25:20,930 - INFO - Driver [/Users/robertmorsch/.wdm/drivers/chromedriver/mac64/127.0.6533.88/chromedriver-mac-x64/chromedriver] found in cache
2024-08-03 10:25:24,157 - INFO - Found 508 snapshots for the channel URL
2024-08-03 10:25:40,387 - INFO - Scraped subscriber count for snapshot: https://web.archive.org/web/20190416052508/https://www.youtube.com/channel/UC1E-JS8L0j1Ei70D9VEFrPQ
2024-08-03 10:25:52,600 - INFO - Scraped subscriber count for snapshot: https://web.archive.org/web/20190416054303/http://www.youtube.com/channel/UC1E-JS8L0j1Ei70D9VEFrPQ
2024-08-03 10:26:05,402 - INFO - Scraped subscriber count for snapshot: https://web.archive.org/web/20190508021050/https://www.youtube.com/channel/UC1E-JS8L0j1Ei70D9VEFrPQ
2024-08-03 10:26:24,612 

                                          snapshot_url           timestamp  \
0    https://web.archive.org/web/20190416052508/htt... 2019-04-16 05:25:08   
1    https://web.archive.org/web/20190416054303/htt... 2019-04-16 05:43:03   
2    https://web.archive.org/web/20190508021050/htt... 2019-05-08 02:10:50   
3    https://web.archive.org/web/20190619220015/htt... 2019-06-19 22:00:15   
4    https://web.archive.org/web/20190703190003/htt... 2019-07-03 19:00:03   
..                                                 ...                 ...   
503  https://web.archive.org/web/20240715061400/htt... 2024-07-15 06:14:00   
504  https://web.archive.org/web/20240715150528/htt... 2024-07-15 15:05:28   
505  https://web.archive.org/web/20240716060254/htt... 2024-07-16 06:02:54   
506  https://web.archive.org/web/20240728152957/htt... 2024-07-28 15:29:57   
507  https://web.archive.org/web/20240802152620/htt... 2024-08-02 15:26:20   

    subscriber_count  
0             12,097  
1             12,